In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
IMG_SIZE = 224
BATCH = 32
HEAD_EPOCHS = 5
FINE_EPOCHS = 15
ROOT = Path("../data")
CLASSES = ["Glioma", "Meningioma", "Pituitary", "No Tumor"]
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

tf.random.set_seed(SEED)
np.random.seed(SEED)
print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPUs available: []


In [2]:
def find_split_dir(name):
    candidates = [
        ROOT / name,
        ROOT / name.capitalize(),
        ROOT / name / name,
        ROOT / name.capitalize() / name.capitalize(),
    ]
    for c in candidates:
        if c.exists() and any((c / cls).exists() for cls in CLASSES):
            return c
    raise FileNotFoundError(f"Could not find class folders for '{name}' under {ROOT.resolve()}")

SPLIT_DIRS = {"train": find_split_dir("train"), "test": find_split_dir("test")}
print(SPLIT_DIRS)

{'train': WindowsPath('../data/train/train'), 'test': WindowsPath('../data/test/test')}


In [3]:
def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

In [4]:
records = []
for split, split_dir in SPLIT_DIRS.items():
    for cls in CLASSES:
        img_dir = split_dir / cls / "images"
        if not img_dir.exists():
            img_dir = split_dir / cls
        for p in img_dir.glob("*.jpg"):
            with Image.open(p) as im:
                w, h = im.size
                mode = im.mode
            records.append({
                "path": str(p), "label": cls, "split": split,
                "width": w, "height": h, "mode": mode, "md5": md5(p),
            })

if not records:
    raise FileNotFoundError(f"No images found under {ROOT.resolve()}")

df = pd.DataFrame(records)
df["is_512"] = ((df["width"] == 512) & (df["height"] == 512)).astype(int)

print("Total images:", len(df))
print("\nImage counts per split and class:")
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))
print("\nShare of 512x512 images per split and class:")
print(df.groupby(["split", "label"])["is_512"].mean().round(3))

Total images: 9257

Image counts per split and class:
label  Glioma  Meningioma  No Tumor  Pituitary
split                                         
test      136         140       100        136
train    3157        3453       711       1424

Share of 512x512 images per split and class:
split  label     
test   Glioma        1.000
       Meningioma    0.907
       No Tumor      0.020
       Pituitary     0.971
train  Glioma        1.000
       Meningioma    0.945
       No Tumor      0.020
       Pituitary     0.971
Name: is_512, dtype: float64


In [5]:
def meta_features(d):
    return np.column_stack([
        d["width"].values, d["height"].values,
        d["is_512"].values, (d["mode"] == "L").astype(int).values,
    ])

train_meta = df[df["split"] == "train"]
test_meta = df[df["split"] == "test"]

baseline = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
baseline.fit(meta_features(train_meta), train_meta["label"])
baseline_pred = baseline.predict(meta_features(test_meta))

print("Metadata-only baseline (no pixel data used)")
print("Accuracy:", round(accuracy_score(test_meta["label"], baseline_pred), 4))
print(classification_report(test_meta["label"], baseline_pred, zero_division=0))

Metadata-only baseline (no pixel data used)
Accuracy: 0.5156
              precision    recall  f1-score   support

      Glioma       0.40      0.38      0.39       136
  Meningioma       0.41      0.84      0.55       140
    No Tumor       0.97      0.92      0.94       100
   Pituitary       0.57      0.03      0.06       136

    accuracy                           0.52       512
   macro avg       0.59      0.54      0.49       512
weighted avg       0.56      0.52      0.45       512



In [6]:
test_hashes = set(df.loc[df["split"] == "test", "md5"])
leak_mask = (df["split"] == "train") & (df["md5"].isin(test_hashes))
print("Train images identical to a test image:", int(leak_mask.sum()))
df = df[~leak_mask]

train_clean = df[df["split"] == "train"].drop_duplicates(subset="md5", keep="first")
test_clean = df[df["split"] == "test"].drop_duplicates(subset="md5", keep="first")
clean = pd.concat([train_clean, test_clean], ignore_index=True)

print("\nImage counts after deduplication:")
print(clean.groupby(["split", "label"]).size().unstack(fill_value=0))

clean.to_csv(OUT_DIR / "clean_manifest.csv", index=False)
print("Saved manifest to", OUT_DIR / "clean_manifest.csv")

Train images identical to a test image: 256

Image counts after deduplication:
label  Glioma  Meningioma  No Tumor  Pituitary
split                                         
test      136         134        93        136
train    3058        2965       508        891
Saved manifest to outputs\clean_manifest.csv


In [7]:
CLASSES_SORTED = sorted(clean["label"].unique())
label_to_idx = {c: i for i, c in enumerate(CLASSES_SORTED)}

train_all = clean[clean["split"] == "train"].reset_index(drop=True)
test_df = clean[clean["split"] == "test"].reset_index(drop=True)
train_df, val_df = train_test_split(
    train_all, test_size=0.15, stratify=train_all["label"], random_state=SEED
)

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))
print("Classes:", CLASSES_SORTED)

y_train = train_df["label"].map(label_to_idx).values
weights = compute_class_weight("balanced", classes=np.arange(len(CLASSES_SORTED)), y=y_train)
class_weight = {i: float(w) for i, w in enumerate(weights)}
print("Class weights:", class_weight)

Train: 6308 | Val: 1114 | Test: 499
Classes: ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
Class weights: {0: 0.6067718353212774, 1: 0.6257936507936508, 2: 3.650462962962963, 3: 2.083223249669749}


In [8]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img, label

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
])

def make_ds(df, training):
    paths = df["path"].values
    labels = df["label"].map(label_to_idx).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_df, training=True)
val_ds = make_ds(val_df, training=False)
test_ds = make_ds(test_df, training=False)
print("Datasets ready")

Datasets ready


In [9]:
base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(CLASSES_SORTED), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.summary(show_trainable=True, expand_nested=False)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Layer (type)                          ┃ Output Shape                  ┃        Param # ┃ Traina… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ input_layer_2 (InputLayer)            │ (None, 224, 224, 3)           │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│ efficientnetb0 (Functional)           │ (None, 7, 7, 1280)            │      4,049,571 │    N    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│ global_average_pooling2d              │ (None, 1280)                  │              0 │    -    │
│ (GlobalAveragePooling2D)              │                               │                │         │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│ dropout (Dropout)                     │ (None, 1280)                  │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│ dense (Dense)                         │ (None, 4)                     │          5,124 │    Y    │
└───────────────────────────────────────┴───────────────────────────────┴────────────────┴─────────┘

 Total params: 4,054,695 (15.47 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [10]:
def make_callbacks(ckpt_path):
    return [
        tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6),
    ]

HEAD_CKPT = OUT_DIR / "head_model.keras"
FINE_CKPT = OUT_DIR / "finetune_model.keras"

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

print("Phase 1: training classification head")
history_head = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
                         class_weight=class_weight, callbacks=make_callbacks(HEAD_CKPT))

Phase 1: training classification head
Epoch 1/5


C:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


198/198 ━━━━━━━━━━━━━━━━━━━━ 385s 2s/step - accuracy: 0.7240 - loss: 0.6229 - val_accuracy: 0.7765 - val_loss: 0.5693 - learning_rate: 0.0010
Epoch 2/5
198/198 ━━━━━━━━━━━━━━━━━━━━ 328s 2s/step - accuracy: 0.8234 - loss: 0.3831 - val_accuracy: 0.8187 - val_loss: 0.4725 - learning_rate: 0.0010
Epoch 3/5
198/198 ━━━━━━━━━━━━━━━━━━━━ 385s 2s/step - accuracy: 0.8513 - loss: 0.3255 - val_accuracy: 0.8232 - val_loss: 0.4283 - learning_rate: 0.0010
Epoch 4/5
198/198 ━━━━━━━━━━━━━━━━━━━━ 1022s 5s/step - accuracy: 0.8581 - loss: 0.3013 - val_accuracy: 0.8411 - val_loss: 0.3992 - learning_rate: 0.0010
Epoch 5/5
198/198 ━━━━━━━━━━━━━━━━━━━━ 2366s 12s/step - accuracy: 0.8630 - loss: 0.2848 - val_accuracy: 0.8528 - val_loss: 0.3837 - learning_rate: 0.0010


In [12]:
model.save("../model/brain_tumor_cnn.keras")